# PROJET MODELISATION GEOMETRIQUE

## SUJET

**Surfacique : nuage de points**

De façon similaire au TP sur les maillages, il sera demandé de pouvoir retrouver une tangente et un rayon en chaque point, ainsi qu’un point sur la surface définie par le nuage de points.

In [108]:
# Imports
import numpy as np
import plotly.graph_objects as go
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from utils import *

# Paramètres
NUM_POINTS = 100                # Nombre de points
NUM_NEIGHBORS = 5               # Nombre de voisin par point
GENERATION_TYPE = 'surface'     # Le type de repartition des points

### Générer le nuage de points

In [109]:
def generate_spherical_points(size, num_points, distribution):

    if distribution == 'surface':
        edge = int(np.sqrt(num_points))
        x = np.linspace(-size, size, edge)
        y = np.linspace(-size, size, edge)
        x, y = np.meshgrid(x, y)
        z = x**2 - y**2
        z *= size/np.max(z)
        points = np.column_stack((x.flatten(), y.flatten(), z.flatten()))

    elif distribution == 'regular_phere':
        indices = np.arange(0, num_points, dtype=float) + 0.5
        phi = np.arccos(1 - 2 * indices/num_points)
        theta = np.pi * (1 + 5**0.5) * indices
        x = size * np.cos(theta) * np.sin(phi)
        y = size * np.sin(theta) * np.sin(phi)
        z = size * np.cos(phi)
        points = np.column_stack((x, y, z))

    elif distribution == 'random_phere':
        theta = 2 * np.pi * np.random.rand(num_points)
        phi = np.arccos(2 * np.random.rand(num_points) - 1)
        x = size * np.sin(phi) * np.cos(theta)
        y = size * np.sin(phi) * np.sin(theta)
        z = size * np.cos(phi)
        points = np.column_stack((x, y, z))
        
    return points


points = generate_spherical_points(5, NUM_POINTS, GENERATION_TYPE)

In [110]:
fig = new_fig()

add_points(points, fig, 'magenta')

fig.show()

### Obtenir les voisins

In [111]:
def find_direct_neighbors(points, num_neighbors):
    nbrs = NearestNeighbors(n_neighbors=num_neighbors + 1, algorithm='auto').fit(points)
    distances, indices = nbrs.kneighbors(points)
    return indices

direct_neighbors = find_direct_neighbors(points, NUM_NEIGHBORS)

In [112]:
for i, point in enumerate(points):
    for neighbor_index in direct_neighbors[i]:
        neighbor_point = points[neighbor_index]
        add_line(point, neighbor_point, fig, 'blue')
        
fig.show()

### Obtenir les k plus proches voisins

In [113]:
def get_k_near_neighbors(neighbors_indices, k):

    k_near_neighbors = [set() for _ in range(NUM_POINTS)]
    
    for point in range(NUM_POINTS):
        k_near_neighbors[point].add(point)

    for _ in range(k):
        for point in range(NUM_POINTS):
            for neighbor in neighbors_indices[point]:
                k_near_neighbors[point].update(neighbors_indices[neighbor])
    
    for point in range(NUM_POINTS):
        k_near_neighbors[point].remove(point)
    
    return k_near_neighbors


k_near_neighbors = get_k_near_neighbors(direct_neighbors, 2)

In [114]:
fig = new_fig()

add_points(points, fig, 'magenta')

add_point(points[0], fig, 'red')

for neighbor_index in k_near_neighbors[0]:
    add_point(points[neighbor_index], fig, 'orange')

fig.show()

### Calculer les normales sortantes

In [115]:
def get_normal(points, neighbors):

    normals = []
    
    for p in range(len(points)):
        neighbor_points = points[list(neighbors[p])]
        
        pca = PCA(n_components=3)
        pca.fit(neighbor_points)
        
        normal_vector = pca.components_[2]
        if np.linalg.norm(points[p] + normal_vector) < np.linalg.norm(points[p]) :
            normal_vector =  - normal_vector
        normals.append(normal_vector)
    
    return normals


normals = get_normal(points, k_near_neighbors)

In [116]:
fig = new_fig()

add_points(points, fig, 'magenta')

for point, normal in zip(points, normals):
    add_line(point, point + normal, fig, 'orange')

fig.show()

### Trouver les rayons

In [117]:
def get_rayons(points, direct_neighbors):
    rayons = []
    for i, voisins in enumerate(direct_neighbors):
        distances = [np.linalg.norm(points[i] - points[j]) for j in voisins]
        mean_distance = np.mean(distances)
        rayons.append(mean_distance / 2.0)
    return rayons

rayons = get_rayons(points, direct_neighbors)

In [118]:
fig = new_fig()

for point, rayon, normal in zip(points, rayons, normals):
    add_circle(point, rayon, normal, fig, 'lightgreen')

fig.show()